# schedule_raw_re_comm_preferences
Fetches `/constituent/v1/constituents/{id}/communicationpreferences` for every
constituent ID in the `raw_re_constituents` Domo dataset.

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb


In [ ]:
DOMO_INPUT_DATASET  = "raw_re_constituents"
                              # ⚠️  This must match the Domo dataset name for UUID
                              #    9b2cc153-7dc1-4e9c-b9e2-6f0ef315ba12.
                              #    Confirm in Domo that dataset is named raw_re_constituents.
ID_COLUMN           = "id"
DOMO_OUTPUT_DATASET = "raw_re_communicationpreferences"  # replace with UUID
COMM_URL_TPL = API_BASE + "/constituent/v1/constituents/{const_id}/communicationpreferences"


In [ ]:
df_consts = domo.read_dataframe(DOMO_INPUT_DATASET, query="SELECT * FROM table")
const_ids = df_consts[ID_COLUMN].dropna().astype(str).str.strip().unique()
print(f"Loaded {len(const_ids):,} constituent IDs")


In [ ]:
token_mgr = TokenManager(interactive=False)
sess      = requests.Session()
results   = []

for i, cid in enumerate(const_ids, 1):
    resp = api_request_with_auth(
        "GET", COMM_URL_TPL.format(const_id=cid),
        token_mgr=token_mgr, session=sess,
    )
    results.append({"constituent_id": cid, "_status": resp.status_code,
                    "data": resp.json() if resp.ok else None})
    if i % 500 == 0:
        print(f"  {i:,}/{len(const_ids):,}")

print(f"Done. {sum(r['_status']==200 for r in results):,} succeeded.")


In [ ]:
rows = []
for r in results:
    if r["_status"] != 200 or not r["data"]:
        continue
    data  = r["data"]
    prefs = data if isinstance(data, list) else data.get("value", [])
    for p in prefs:
        if isinstance(p, dict):
            p["constituent_id"] = r["constituent_id"]
            rows.append(p)

df_comm = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["constituent_id"])
print(f"Flattened to {len(df_comm):,} rows")


In [ ]:
df_out = domo_safe_cast(df_comm)
domo.write_dataframe(df_out, dataset=DOMO_OUTPUT_DATASET, update_method="REPLACE")
print(f"✅ Wrote {len(df_out):,} rows → {DOMO_OUTPUT_DATASET}")
